## Imports e Configurações

In [ ]:
import os
import time
import re
import pandas as pd
import urllib
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

import requests
import fitz  # PyMuPDF
import json

# --- CONFIGURAÇÕES DE DIRETÓRIOS ---
# Mapeamento de tipos para subpastas conforme sua organização
tipos_normas = {
    "Resolução CMN": "cmn",
    "Resolução BCB": "bcb",
    "Instrução Normativa BCB": "in_bcb",

    "Resolução Coseg": "res_coseg",
    "Resolução Coremec": "res_coremec",

    "Carta Circular": "cart_circular",
    "Circular": "circular"
}

# --- CONFIGURAÇÃO DE PDF ---
# Dicionário para conversão de meses por extenso
MESES = {
    "janeiro": "1", "fevereiro": "2", "março": "3", "abril": "4",
    "maio": "5", "junho": "6", "julho": "7", "agosto": "8",
    "setembro": "9", "outubro": "10", "novembro": "11", "dezembro": "12"
}

for subpasta in tipos_normas.values():
    os.makedirs(f"normas_bacen/{subpasta}", exist_ok=True)
    os.makedirs(f"normas_bacen_tratadas/{subpasta}", exist_ok=True)

# --- CONFIGURAÇÃO DO DRIVER ---
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=chrome_options)

# --- CSV DE CONTROLE ---
CSV_CONTROLE = "controle_extracao.csv"

## Funções

In [ ]:
# ------------------------------------------
# === FUNÇÕES AUXILIARES ===
# ------------------------------------------
def limpar_texto(texto):
    """
    Tratamento avançado para manter a integridade semântica para a IA.
    """
    # Normalização de hifens especiais e quebras de sistema
    texto = texto.replace('\r\n', '\n').replace('\xad', '-')
    texto = re.sub(r'[ \t]+', ' ', texto)
    # Une quebras que cortam citações externas no meio da frase
    texto = re.sub(r'(no|art\.|inciso|§)\s*\n+\s*', r'\1 ', texto, flags=re.IGNORECASE)
    # Proteção de início de linha para estrutura real da norma (Artigos e Parágrafos)[cite: 12, 17]
    padrao_estrutura = r'(\n(?=(Art\.|§|Parágrafo|RESOLUÇÃO|INSTRUÇÃO|Estabelece|O Banco|[IVX]+\s?-|[a-z]\)\s?|“Art\.)))'
    texto = re.sub(padrao_estrutura, '[[PARAGRAFO]]', texto)
    # Remove quebras simples e restaura parágrafos duplos
    texto = re.sub(r'(?<!\[\[PARAGRAFO\]\])\n(?!\[\[PARAGRAFO\]\])', ' ', texto)
    texto = texto.replace('[[PARAGRAFO]]', '\n\n')
    # Limpa sequências de pontos de edição (comuns em alterações de normas)
    texto = re.sub(r'\.{4,}', '', texto)
    # Remove quebras de linha extras (transforma duas ou mais em apenas uma)
    texto = re.sub(r'\n{2,}', '\n', texto)
    return re.sub(r' +', ' ', texto).strip()

def reiniciar_driver():
    """Fecha a instância atual e inicia um novo navegador para limpar o cache."""
    global driver
    print("\n[Estabilidade] Reiniciando Chromedriver para evitar erros de conexão...")
    try:
        driver.quit()
    except:
        pass
    time.sleep(2)
    driver = webdriver.Chrome(options=chrome_options)

def atualizar_controle(tipo, numero, status):
    """
    Registra ou atualiza o status de uma norma no arquivo CSV.
    """
    # Mapeia o nome completo para a sigla da sua coluna
    mapa_siglas = {
        "Resolução CMN": "CMN",
        "Resolução BCB": "BCB",
        "Instrução Normativa BCB": "IN_BCB",
        "Resolução Coseg": "RES_COSEG",
        "Resolução Coremec": "RES_COREMEC",
        "Carta Circular": "CART_CIRCULAR",
        "Circular": "CIRCULAR"
    }
    sigla = mapa_siglas.get(tipo, tipo)
    data_hoje = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    
    if os.path.exists(CSV_CONTROLE):
        df = pd.read_csv(CSV_CONTROLE)
    else:
        # Cria o dataframe inicial se o arquivo não existir
        df = pd.DataFrame(columns=['tipo', 'numero', 'status', 'data_extracao'])

    # Verifica se a norma já existe no CSV para atualizar ou adicionar
    mascara = (df['tipo'] == sigla) & (df['numero'] == int(numero))
    
    if not df[mascara].empty:
        df.loc[mascara, 'status'] = status
        df.loc[mascara, 'data_extracao'] = data_hoje
    else:
        novo_registro = {
            'tipo': sigla, 
            'numero': int(numero), 
            'status': status, 
            'data_extracao': data_hoje
        }
        df = pd.concat([df, pd.DataFrame([novo_registro])], ignore_index=True)

    df.to_csv(CSV_CONTROLE, index=False, encoding='utf-8-sig')


# ------------------------------------------
# === SCRAPPING ===
# ------------------------------------------
def processar_norma(tipo_nome, numero):
    subpasta = tipos_normas.get(tipo_nome)
    tipo_encoded = urllib.parse.quote(tipo_nome)
    url = f"https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo={tipo_encoded}&numero={numero}"
    
    print(f"\nProcessando [{tipo_nome}] {numero}...")
    
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 30)

        # 1. SINCRONIZAÇÃO
        seletor_redundante = "#conteudoTexto, .corpoNormativo, .container.main"
        wait.until(lambda d: d.find_element(By.CSS_SELECTOR, seletor_redundante))

        # 2. EXTRAÇÃO DA DATA
        situacao = "EM VIGOR"
        data_vigencia = "Não encontrada"
        titulos = driver.find_elements(By.TAG_NAME, "h2")
        if titulos:
            busca_data = re.search(r'(\d{1,2}/\d{1,2}/\d{4})', titulos[0].text)
            if busca_data: data_vigencia = busca_data.group(1)
            if "REVOGADO" in titulos[0].text.upper(): situacao = "REVOGADA"

        # 3. EXTRAÇÃO DO CONTEÚDO
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        corpo = soup.select_one(".corpoNormativo, #conteudoTexto, #conteudoCancelado, .WordSection1")
        
        texto_vigente_base = ""

        # LÓGICA DE PRIORIDADE: PDF vs HTML
        # Se não houver corpo ou o texto for muito curto, busca PDF
        if not corpo or len(corpo.get_text().strip()) < 50:
            print("  - [LOG] Conteúdo HTML insuficiente. Tentando baixar PDF...")
            try:
                url = f"https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o&numero={numero}"
                driver.get(url)

                soup = BeautifulSoup(driver.page_source, 'html.parser')

                # Captura o Assunto com seletor robusto
                assunto_div = soup.select_one(".assunto")
                if assunto_div:
                    # Remove o texto do h3 (título "Assunto") para pegar apenas a descrição
                    for h3 in assunto_div.find_all("h3"):
                        h3.decompose()
                    assunto_texto = assunto_div.get_text().strip()
                else:
                    assunto_texto = "Assunto não disponível"

                wait_pdf = WebDriverWait(driver, 15)
                # Seletor CSS robusto para qualquer link PDF na lista
                link_pdf_tag = wait_pdf.until(EC.element_to_be_clickable((
                    By.CSS_SELECTOR, "li.list-group-item a[href*='.pdf']"
                )))
                
                url_pdf = link_pdf_tag.get_attribute("href")
                nome_pdf = f"{subpasta}_{numero}.pdf"
                caminho_pdf = f"normas_bacen/{subpasta}/{nome_pdf}"
                
                # Download binário
                r = requests.get(url_pdf, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
                r.raise_for_status()
                with open(caminho_pdf, 'wb') as f:
                    f.write(r.content)
                
                texto_vigente_base = f"ASSUNTO: {assunto_texto}]"
                
            except Exception as e:
                print(f"  - [AVISO] PDF não encontrado. Usando assunto disponível.")
                texto_vigente_base = f"ASSUNTO: {assunto_texto}\n\n[CONTEÚDO INTEGRAL INDISPONÍVEL NO PORTAL]"
        
        # LÓGICA 2: Extração via HTML (se o texto do PDF não foi capturado)
        if not texto_vigente_base and corpo:
            print(f"  - [LOG] Extraindo via HTML (ID: {corpo.get('id')})")
            for revogado in corpo.find_all(['strike', 's', 'del']):
                revogado.decompose()
            
            # Garante que o Assunto apareça no início do TXT
            texto_vigente_base = f"ASSUNTO: {assunto_texto}\n\n" + corpo.get_text(separator='\n').strip()
            
            if corpo.get('id') == "conteudoCancelado":
                situacao = "CANCELADO"

        # 4. SALVAMENTO (Unificado)
        if texto_vigente_base:
            if "REVOGADA" in texto_vigente_base[:150].upper():
                situacao = "REVOGADA"

            nome_arquivo = f"{subpasta}_{numero}.txt"
            cabecalho = f"[URL]: {url}\n[FILE_ID]: {nome_arquivo}\n[DT_VGNCIA]: {data_vigencia}\n[SITUACAO]: {situacao}\n\n"
            
            # Salva Bruto e Tratado conforme sua estrutura de pastas
            with open(f"normas_bacen/{subpasta}/{nome_arquivo}", "w", encoding="utf-8") as f:
                f.write(cabecalho + texto_vigente_base)
            
            texto_tratado = limpar_texto(texto_vigente_base)
            with open(f"normas_bacen_tratadas/{subpasta}/{nome_arquivo}", "w", encoding="utf-8") as f:
                f.write(cabecalho + texto_tratado)
            
            return True
        
        return False

    except Exception as e:
        print(f"  - [LOG] Erro em [{tipo_nome}] {numero}: {str(e)}")
        return False
    

# ------------------------------------------
# === PDFs PARA .TXT ===
# ------------------------------------------
def extrair_data_cabecalho(texto_limpo):
    """ Extrai a data do cabeçalho da norma """
    padrao = re.search(r"(RESOLUÇÃO|INSTRUÇÃO)\s+Nº\s+[\d\.]+,?\s+DE\s+(\d{1,2})\s+DE\s+([a-zA-Zç]+)\s+DE\s+(\d{4})", texto_limpo, re.IGNORECASE)
    if padrao:
        dia = padrao.group(2)
        mes_extenso = padrao.group(3).lower()
        ano = padrao.group(4)
        mes_num = MESES.get(mes_extenso, "01")
        return f"{dia}/{mes_num}/{ano}"
    return "Não encontrada"

def extrair_dados_pdf(caminho_pdf):
    doc = fitz.open(caminho_pdf)
    blocos_validos = []
    texto_completo_com_riscados = "" # Usado para verificar a revogação

    for pagina in doc:
        # Captura todo o texto da página (incluindo riscados) para verificar revogação
        texto_completo_com_riscados += pagina.get_text()
        
        desenhos = pagina.get_drawings()
        # Filtra textos riscados (strikethrough)
        linhas_riscado = [
            d["items"][0][1:] for d in desenhos 
            if d["items"][0][0] == "l" and abs(d["items"][0][1].y - d["items"][0][2].y) < 1
        ]
        
        dict_texto = pagina.get_text("dict")
        for bloco in dict_texto["blocks"]:
            if "lines" in bloco:
                for linha in bloco["lines"]:
                    texto_linha = ""
                    for span in linha["spans"]:
                        span_rect = fitz.Rect(span["bbox"])
                        # Verifica se o texto está riscado geometricamente[cite: 1]
                        if not any(span_rect.contains(p1) or span_rect.contains(p2) for p1, p2 in linhas_riscado):
                            texto_linha += span["text"]
                    
                    if texto_linha.strip():
                        blocos_validos.append(texto_linha)

    # Verifica se o termo "normativo revogado" existe em qualquer lugar do documento[cite: 1]
    is_revogada = "normativo revogado" in texto_completo_com_riscados.lower()
    
    texto_bruto = "\n".join(blocos_validos)
    texto_limpo = limpar_texto(texto_bruto) # Assume-se que a função limpar_texto existe no seu ambiente
    data_vgncia = extrair_data_cabecalho(texto_limpo)
    
    doc.close()
    return data_vgncia, texto_limpo, is_revogada

def processar_arquivos(diretorio_origem, diretorio_destino):
    if not os.path.exists(diretorio_destino):
        os.makedirs(diretorio_destino)

    tags_para_manter = ["[URL]:", "[FILE_ID]:"] # SITUACAO agora é tratada separadamente

    for arq in os.listdir(diretorio_origem):
        if arq.endswith(".pdf"):
            caminho_pdf = os.path.join(diretorio_origem, arq)
            nome_txt = arq.replace(".pdf", ".txt")
            caminho_txt_destino = os.path.join(diretorio_destino, nome_txt)
            
            if os.path.exists(caminho_txt_destino):
                data_vgncia, conteudo, is_revogada = extrair_dados_pdf(caminho_pdf)
                
                with open(caminho_txt_destino, 'r', encoding='utf-8') as f:
                    linhas = f.readlines()
                
                novas_linhas = []
                for linha in linhas:
                    # 1. Atualiza Data de Vigência[cite: 1]
                    if linha.startswith("[DT_VGNCIA]:"):
                        novas_linhas.append(f"[DT_VGNCIA]: {data_vgncia}\n")
                    
                    # 2. Atualiza Situação
                    elif linha.startswith("[SITUACAO]:"):
                        status = "REVOGADA" if is_revogada else "EM VIGOR"
                        novas_linhas.append(f"[SITUACAO]: {status}\n")
                    
                    # 3. Ignora as tags que devem ser removidas
                    elif linha.startswith("ASSUNTO:") or "[CONTEÚDO NO PDF ANEXO:" in linha:
                        continue
                    
                    # 4. Mantém apenas os metadados específicos
                    else:
                        for tag in tags_para_manter:
                            if tag in linha:
                                novas_linhas.append(linha)
                                break

                # 5. Reconstrói o arquivo
                final_txt = "".join(novas_linhas).strip()
                final_txt += f"\n\n{conteudo}"
                
                with open(caminho_txt_destino, 'w', encoding='utf-8') as f:
                    f.write(final_txt)
                
                print(f"Processado: {nome_txt} | Status: {'REVOGADA' if is_revogada else 'EM VIGOR'}")


# ------------------------------------------
# === MAP DE TEMAS DOS NORMATIVOS ===
# ------------------------------------------
def mapear_temas_do_html(html_content):
    """
    Analisa o HTML buscando os títulos h6 e associando-os aos links 
    da lista ul subsequente.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    lista_temas_mapeados = []
    
    # Encontramos todos os títulos de categoria
    categorias_h6 = soup.find_all('h6', class_='titulo-tema')
    
    for h6 in categorias_h6:
        # Extrai o nome da categoria (ex: Aplicações financeiras, Cadastros...)
        categoria_nome = h6.get_text(strip=True)
        
        # Encontra a lista 'ul' que está logo após este h6 específico
        ul_correspondente = h6.find_next_sibling('ul')
        
        if ul_correspondente:
            links = ul_correspondente.find_all('a')
            for link in links:
                nome_subtema = link.get_text(strip=True)
                href = link.get('href', '')
                
                # Extração do ID do tema via regex
                tema_id_match = re.search(r'tema=(\d+)', href)
                
                if tema_id_match:
                    lista_temas_mapeados.append({
                        "categoria": categoria_nome,
                        "subtema_nome": nome_subtema,
                        "tema_id": tema_id_match.group(1)
                    })
    
    return lista_temas_mapeados

def extrair_normas_por_tema(driver, tema_info):
    tema_id = tema_info['tema_id']
    nome_tema = tema_info['subtema_nome']
    
    base_url = f"https://www.bcb.gov.br/estabilidadefinanceira/buscanormas?tema={tema_id}"
    driver.get(base_url)
    
    todas_normas_tema = []
    
    while True:
        try:
            # Aguarda a lista de resultados carregar
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CLASS_NAME, "resultado-item"))
            )
            
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            itens = soup.find_all('li', class_='resultado-item')
            
            for item in itens:
                link_tag = item.find('a')
                if not link_tag: continue
                
                titulo_completo = link_tag.get_text(strip=True)
                
                # --- NOVO: Extração de Tipo e Número via Regex ---
                # Procura o padrão: [Texto] n° [Número]
                match = re.search(r'^(.*?)\s+n°\s+([\d.]+)', titulo_completo)
                
                if match:
                    tipo_norma = match.group(1).strip()
                    # Remove pontos (ex: 5.182 -> 5182) para compatibilidade com o scraper principal
                    numero_norma = match.group(2).replace('.', '')
                else:
                    tipo_norma = "Não identificado"
                    numero_norma = "N/A"
                
                # Extração segura da data
                texto_completo = item.get_text()
                data_doc = "N/A"
                if "Data/Hora Documento:" in texto_completo:
                    data_doc = texto_completo.split("Data/Hora Documento:")[1].strip().split()[0]
                
                norma = {
                    "titulo": titulo_completo,
                    "tipo_norma": tipo_norma,      # Adicionado
                    "numero_norma": numero_norma,  # Adicionado
                    "url_exibicao": "https://www.bcb.gov.br" + link_tag['href'],
                    "assunto": item.find('span').get_text(strip=True) if item.find('span') else "",
                    "data_documento": data_doc
                }
                todas_normas_tema.append(norma)
            
            # Paginação
            try:
                botao_proximo = driver.find_element(By.CSS_SELECTOR, "li.page-item:not(.disabled) a[aria-label='Próxima']")
                driver.execute_script("arguments[0].click();", botao_proximo)
                time.sleep(2) 
            except:
                break # Não há mais páginas
                
        except Exception as e:
            print(f"Fim das páginas ou erro no tema {nome_tema}: {e}")
            break
            
    return todas_normas_tema

## Execução: Scrapping

In [ ]:
# --- CONFIGURAÇÃO ---
config_execucao = [

    {"tipo": "Resolução CMN", "inicio": 4839, "fim": 1},

    {"tipo": "Circular", "inicio": 4038, "fim": 1},
    {"tipo": "Carta Circular", "inicio": 4076, "fim": 1},

    {"tipo": "Resolução Coseg", "inicio": 1, "fim": 1},
    {"tipo": "Resolução Coremec", "inicio": 1, "fim": 1},

    {"tipo": "Resolução BCB", "inicio": 563, "fim": 1},
    {"tipo": "Instrução Normativa BCB", "inicio": 731, "fim": 1}
]

# Contador para gestão de memória e estabilidade
contador_estabilidade = 0

# --- CONFIGURAÇÃO ---
for config in config_execucao:
    tipo = config["tipo"]
    subpasta = tipos_normas[tipo]
    inicio = config["inicio"]
    fim = config["fim"]
    
    passo = 1 if fim > inicio else -1
    numeros = range(inicio, fim + passo, passo)
    
    print(f"\n--- Iniciando varredura de {tipo} ({'Crescente' if passo == 1 else 'Decrescente'}) ---")
    
    tentativas_vazias = 0
    
    for num in numeros:
        if contador_estabilidade >= 20:
            reiniciar_driver()
            contador_estabilidade = 0

        if passo == 1 and tentativas_vazias >= 3:
            print(f"Interrompendo {tipo}: 3 sequências sem resultados.")
            break
            
        nome_arquivo = f"{subpasta}_{num}.txt"
        caminho_bruto = f"normas_bacen/{subpasta}/{nome_arquivo}"
        
        # [PONTO 1] CHECKPOINT: Se já existe, marca OK no CSV e pula
        if os.path.exists(caminho_bruto):
            atualizar_controle(tipo, num, "OK") # <--- AQUI
            tentativas_vazias = 0
            continue
            
        sucesso = False
        try:
            sucesso = processar_norma(tipo, num)
            contador_estabilidade += 1
        except Exception as e:
            print(f"Erro detectado na norma {num}. Tentando recuperar conexão...")
            print(f"[ERRO]: {e}")
            atualizar_controle(tipo, num, "ERRO") # <--- AQUI (Opcional, pois tentará de novo)
            reiniciar_driver()
            try:
                sucesso = processar_norma(tipo, num)
                contador_estabilidade += 1
            except:
                print(f"Falha persistente na norma {num}. Pulando...")
                atualizar_controle(tipo, num, "ERRO") # <--- AQUI

        if sucesso:
            # [PONTO 2] SUCESSO: Download e tratamento feitos
            atualizar_controle(tipo, num, "OK") # <--- AQUI
            tentativas_vazias = 0
            time.sleep(3)
        else:
            # [PONTO 3] FALHA: Norma não existe ou está revogada
            # Só marcamos como PENDENTE se não for um erro de código
            atualizar_controle(tipo, num, "PENDENTE") # <--- AQUI
            tentativas_vazias += 1
            if passo == 1:
                print(f"Norma {num} não encontrada ({tentativas_vazias}/3)")

# Encerramento seguro
try:
    driver.quit()
except:
    pass

print("\nProcesso de sincronização concluído com sucesso.")

## Execução: PDF -> TXT

In [ ]:
origem = r'normas_bacen\cmn'
destino = r'normas_bacen_tratadas\cmn'
processar_arquivos(origem, destino)

## Execução: Temas JSON

In [ ]:
html_temas = """
<div _ngcontent-uob-c33="" id="containerTemasRelevantes" class="container collapse"><div _ngcontent-uob-c33="" class="row pt-3 pb-3"><div _ngcontent-uob-c33="" class="col-sm-6"><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Aplicações financeiras</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=1">CDB/RDB</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=2">Poupança (abertura)</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=3">Poupança (Remuneração)</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Arranjos de pagamentos</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=4">Arranjos de pagamentos</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Cadastros</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=5">Cadastro Positivo</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=6">CCF</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=7">CCS</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=8">SCR</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Câmbio e capitais internacionais</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=9">Capital estrangeiro no país</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=10">Capital brasileiro no exterior</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=11">Convênio de Pagamentos e Créditos Recíprocos (CCR)</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=12">Exportação e Importação</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=13">Operações de câmbio - turistas</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=14">Taxa de câmbio</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Cartão de crédito</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=15">Cartão de crédito</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Cheques</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=35">Devolução</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=36">Compensação</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Consórcios</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=37">Consórcios</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Contas</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=16">Abertura, encerramento e movimentação</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=17">Conta simplificada</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=18">Conta salário</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Cooperativas de crédito</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=19">Cooperativas de crédito</a></li><!----></ul></div><!----></div><div _ngcontent-uob-c33="" class="col-sm-6"><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Correspondentes no país</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=20">Correspondentes (lotéricas, lojas de conveniência, Banco Postal)</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Empréstimos e Financiamentos</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=21">Custo Efetivo Total (CET)</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=22">Liquidação antecipada</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=23">Portabilidade de crédito</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=24">Portabilidade cadastral</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Fundo Garantidor de Créditos</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=25">Fundo Garantidor de Créditos (FGC)</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=26">Fundo Garantidor do Cooperativismo de Crédito (FGCoop)</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Lavagem de dinheiro</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=27">Lavagem de dinheiro</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Leasing - Arrendamento mercantil</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=28">Arrendamento mercantil</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Moedas e cédulas</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=29">Moedas e cédulas</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Ouvidoria</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=30">Ouvidoria</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Pagamentos de contas e transferências de crédito</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=31">Pagamentos de contas e transferências de crédito</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Taxas de juros</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=32">Taxa Selic</a></li><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=33">Taxa Referencial - TR</a></li><!----></ul></div><div _ngcontent-uob-c33=""><h6 _ngcontent-uob-c33="" class="titulo-tema mb-1">Tarifas</h6><ul _ngcontent-uob-c33="" class="list-unstyled"><li _ngcontent-uob-c33=""><a _ngcontent-uob-c33="" href="/estabilidadefinanceira/buscanormas?tema=34">Tarifas</a></li><!----></ul></div><!----></div></div></div>
"""

# Mapeamento Automático
temas_mapeados = mapear_temas_do_html(html_temas)
mapa_final = {}
total_temas = len(temas_mapeados)

print(f"Total de subtemas encontrados: {total_temas}\n")

# Loop de Extração
for i, tema in enumerate(temas_mapeados, 1):
    cat = tema['categoria']
    sub = tema['subtema_nome']
    tid = tema['tema_id']
    
    print(f"[{i}/{total_temas}] Categoria: {cat} | Subtema: {sub}")
    
    # Cria a chave da categoria no dicionário se ela não existir
    if cat not in mapa_final:
        mapa_final[cat] = {}
    
    # Chama sua função de extração (extrair_normas_por_tema)
    normas_extraidas = extrair_normas_por_tema(driver, tema)
    
    # Armazena os dados estruturados
    mapa_final[cat][sub] = {
        "tema_id": tid,
        "total_normas": len(normas_extraidas),
        "normas": normas_extraidas
    }

# Salva o JSON final
with open("mapeamento_normas_por_tema.json", "w", encoding="utf-8") as f:
    json.dump(mapa_final, f, indent=4, ensure_ascii=False)

print("\nJSON gerado com sucesso")